# Crop Recommendation Model Training
**Model:** Random Forest Classifier (Multi-class)
**Output:** `backend/app/ml_models/crop_model.pkl`

**Dataset:** [Kaggle Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)
Download `Crop_recommendation.csv` and place it in `data/raw/`

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Load dataset
df = pd.read_csv('../data/raw/Crop_recommendation.csv')
print(df.head())
print(f'Shape: {df.shape}')
print(f'Crops: {df["label"].unique()}')

In [ ]:
# The Kaggle dataset has: N, P, K, temperature, humidity, ph, rainfall, label
# We map this to our feature set: state_enc, soil_enc, temperature, humidity, rainfall
# For MVP we use the available features directly

FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
TARGET = 'label'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.plot(kind='bar', title='Feature Importances', figsize=(8, 4))
plt.tight_layout()
plt.show()

In [ ]:
# Save model
import os
os.makedirs('../backend/app/ml_models', exist_ok=True)
joblib.dump(model, '../backend/app/ml_models/crop_model.pkl')
print('crop_model.pkl saved!')

# Quick test
loaded = joblib.load('../backend/app/ml_models/crop_model.pkl')
sample = X_test.iloc[:3]
proba = loaded.predict_proba(sample)
top3 = np.argsort(proba, axis=1)[:, ::-1][:, :3]
for i, row in enumerate(top3):
    print(f'Sample {i+1}: Top 3 crops = {[loaded.classes_[j] for j in row]}')